# Skin Disease Detection with Explainability using YOLO11 and Grad-CAM

Eight-class dermoscopic skin lesion classification on the ISIC 2019 dataset, using YOLO11n with Grad-CAM explainability.

Accepted for oral presentation, IEEE ICEC2NT 2026 (Paper ID 2725).

**Author:** Harmanjit Singh Multani · harmanjeet.multani@gmail.com
**Co-authors:** Sandeep Kaur Multani, Atiya Khan

**Pipeline:** Setup → Data loading & stratified split → Training → Evaluation → Per-class analysis → Grad-CAM explainability → Statistical summary


## 1. Setup

Mount Google Drive and install dependencies. This notebook expects the ISIC 2019 training archive and ground-truth CSV to already be uploaded to `MyDrive/skin_research/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install ultralytics -q
!pip install grad-cam -q
!pip install scikit-learn matplotlib seaborn pandas -q

print("✅ Environment ready")


In [ ]:
import os
import shutil
import zipfile
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from ultralytics import YOLO

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

DRIVE_ROOT = '/content/drive/MyDrive/skin_research'
CLASSES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']


## 2. Data Loading, Stratified Split, and Folder Preparation

- Extracts the ISIC 2019 training archive
- Assigns single-label class per image from the multi-hot ground-truth CSV
- Stratified 70/20/10 train/val/test split (seed=42), preserving natural class distribution — **no class balancing applied**, so the model is evaluated under realistic clinical prevalence
- Copies images into `class/split` folders for `ultralytics` classification training


In [ ]:
zip_path = f'{DRIVE_ROOT}/ISIC_2019_Training_Input.zip'
extract_path = '/content/ISIC_2019'
os.makedirs(extract_path, exist_ok=True)

print("Extracting archive (5–10 min)...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

img_dir = f'{extract_path}/ISIC_2019_Training_Input'
print(f"✅ Extracted. Total images: {len(os.listdir(img_dir))}")


In [ ]:
df = pd.read_csv(f'{DRIVE_ROOT}/ISIC_2019_Training_GroundTruth.csv')
df['label'] = df[CLASSES].idxmax(axis=1)
df['class_id'] = df['label'].apply(lambda x: CLASSES.index(x))

print("Class distribution:")
print(df['label'].value_counts())
print(f"\nTotal samples: {len(df)}")


In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.33, random_state=42, stratify=temp_df['label']
)

print(f"✅ Train: {len(train_df)} images")
print(f"✅ Val:   {len(val_df)} images")
print(f"✅ Test:  {len(test_df)} images")


In [ ]:
SRC_DIR = f'{extract_path}/ISIC_2019_Training_Input'
BASE_DIR = '/content/skin_dataset'

for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        os.makedirs(f'{BASE_DIR}/{split}/{cls}', exist_ok=True)

def copy_images(split_df, split_name):
    count = 0
    for _, row in split_df.iterrows():
        img_name = row['image'] + '.jpg'
        src = os.path.join(SRC_DIR, img_name)
        dst = f'{BASE_DIR}/{split_name}/{row["label"]}/{img_name}'
        if os.path.exists(src):
            shutil.copy(src, dst)
            count += 1
    print(f"✅ {split_name}: {count} images copied")

copy_images(train_df, 'train')
copy_images(val_df, 'val')
copy_images(test_df, 'test')
print("✅ Dataset ready!")


## 3. Training

YOLO11n classification head, trained for 50 epochs on the imbalanced natural distribution (AdamW, automatic optimizer selection, 224×224 input, batch size auto).


In [ ]:
model = YOLO('yolo11n-cls.pt')  # ImageNet-pretrained

results = model.train(
    data=BASE_DIR,
    epochs=50,
    imgsz=224,
    project=f'{DRIVE_ROOT}/runs',
    name='yolo11_skin',
)

print("✅ Training complete!")


## 4. Evaluation on Held-Out Test Set

Reports Top-1 / Top-5 accuracy on the untouched 2,508-image test split.


In [ ]:
model = YOLO(f'{DRIVE_ROOT}/runs/yolo11_skin/weights/best.pt')

metrics = model.val(data=BASE_DIR, split='test')

print("=" * 40)
print("TEST SET RESULTS")
print("=" * 40)
print(f"Top-1 Accuracy: {metrics.top1 * 100:.2f}%")
print(f"Top-5 Accuracy: {metrics.top5 * 100:.2f}%")


### Per-image predictions, classification report, and confusion matrix

In [ ]:
all_image_paths, all_true_labels = [], []
for cls in CLASSES:
    cls_folder = f'{BASE_DIR}/test/{cls}'
    for img_name in os.listdir(cls_folder):
        all_image_paths.append(os.path.join(cls_folder, img_name))
        all_true_labels.append(CLASSES.index(cls))

print(f"Total test images: {len(all_image_paths)}")

y_true, y_pred = all_true_labels, []
batch_size = 100
for i in range(0, len(all_image_paths), batch_size):
    batch = all_image_paths[i:i + batch_size]
    batch_results = model.predict(source=batch, save=False, verbose=False)
    y_pred.extend(r.probs.top1 for r in batch_results)

print(classification_report(y_true, y_pred, target_names=CLASSES))


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.title('Confusion Matrix — YOLO11 Skin Disease Detection', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/confusion_matrix.png', dpi=300)
plt.show()


## 5. Dataset Visualization

Sample images per class, and the raw class distribution (illustrating the 54:1 NV:DF imbalance).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, cls in enumerate(CLASSES):
    cls_folder = f'{BASE_DIR}/train/{cls}'
    sample_img = os.listdir(cls_folder)[0]
    img = cv2.imread(os.path.join(cls_folder, sample_img))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    row, col = idx // 4, idx % 4
    axes[row][col].imshow(img)
    axes[row][col].set_title(cls, fontsize=14, fontweight='bold')
    axes[row][col].axis('off')

plt.suptitle('ISIC 2019 — Sample Images Per Disease Class', fontsize=16)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/dataset_samples.png', dpi=300)
plt.show()


In [ ]:
counts = [df[df['label'] == c].shape[0] for c in CLASSES]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22', '#34495e']

plt.figure(figsize=(12, 6))
bars = plt.bar(CLASSES, counts, color=colors)
for bar, val in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100, str(val), ha='center', fontsize=11)

plt.title('Class Distribution — ISIC 2019 Dataset (54:1 NV:DF imbalance)', fontsize=14)
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/class_distribution.png', dpi=300)
plt.show()


## 6. Per-Class Accuracy

In [ ]:
per_class_correct = np.diag(cm)
per_class_total = cm.sum(axis=1)
per_class_acc = [c / t * 100 for c, t in zip(per_class_correct, per_class_total)]

plt.figure(figsize=(12, 6))
bars = plt.bar(CLASSES, per_class_acc, color=colors)
for bar, val in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f'{val:.1f}%', ha='center', fontsize=11)

overall_acc = metrics.top1 * 100
plt.axhline(y=overall_acc, color='red', linestyle='--', label=f'Overall Accuracy {overall_acc:.2f}%')
plt.title('Per-Class Accuracy — YOLO11 Skin Disease Detection', fontsize=14)
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.ylim(0, 110)
plt.legend()
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/per_class_accuracy.png', dpi=300)
plt.show()


In [ ]:
precision = precision_score(y_true, y_pred, average=None, zero_division=0)
recall = recall_score(y_true, y_pred, average=None, zero_division=0)
f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

x = np.arange(len(CLASSES))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width, precision, width, label='Precision', color='#3498db')
ax.bar(x, recall, width, label='Recall', color='#e74c3c')
ax.bar(x + width, f1, width, label='F1-Score', color='#2ecc71')

ax.set_title('Precision, Recall & F1-Score Per Class', fontsize=14)
ax.set_xlabel('Disease Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(CLASSES)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/precision_recall_f1.png', dpi=300)
plt.show()


## 7. Training Curves

In [ ]:
results_csv = pd.read_csv(f'{DRIVE_ROOT}/runs/yolo11_skin/results.csv')
results_csv.columns = results_csv.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(results_csv['epoch'], results_csv['train/loss'], color='#e74c3c', linewidth=2, label='Train Loss')
axes[0].set_title('Training Loss Curve', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(results_csv['epoch'], results_csv['metrics/accuracy_top1'], color='#3498db', linewidth=2, label='Top-1 Accuracy')
axes[1].plot(results_csv['epoch'], results_csv['metrics/accuracy_top5'], color='#2ecc71', linewidth=2, label='Top-5 Accuracy')
axes[1].set_title('Validation Accuracy Curve', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('YOLO11 Training Progress — Skin Disease Detection', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/training_curves.png', dpi=300)
plt.show()


## 8. Explainability — Grad-CAM

Applies Grad-CAM to the final convolutional layer to verify the model attends to lesion morphology rather than imaging artifacts, per the ABCD dermoscopic criteria (Asymmetry, Border irregularity, Color variegation, Diameter).


In [ ]:
pytorch_model = model.model
pytorch_model.eval()
target_layer = [pytorch_model.model[-2]]
cam = GradCAM(model=pytorch_model, target_layers=target_layer)

def load_for_cam(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_float = np.float32(img_rgb) / 255.0
    input_tensor = torch.from_numpy(img_float).permute(2, 0, 1).unsqueeze(0).cuda().requires_grad_(True)
    return img_rgb, img_float, input_tensor


In [ ]:
# Single-image example (melanoma)
sample_folder = f'{BASE_DIR}/test/MEL'
sample_path = os.path.join(sample_folder, os.listdir(sample_folder)[0])
img_rgb, img_float, input_tensor = load_for_cam(sample_path)

grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(0)])
visualization = show_cam_on_image(img_float, grayscale_cam[0], use_rgb=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_rgb); axes[0].set_title('Original Image', fontsize=13); axes[0].axis('off')
axes[1].imshow(visualization); axes[1].set_title('Grad-CAM Explanation', fontsize=13); axes[1].axis('off')
plt.suptitle('Explainability — YOLO11 Skin Disease Detection (Melanoma)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/gradcam.png', dpi=300)
plt.show()


In [ ]:
# Grid across all 8 classes
test_image_paths = []
for cls in CLASSES:
    cls_folder = f'{BASE_DIR}/test/{cls}'
    imgs = os.listdir(cls_folder)
    if imgs:
        test_image_paths.append(os.path.join(cls_folder, imgs[0]))

fig, axes = plt.subplots(2, len(test_image_paths), figsize=(20, 7))
for i, img_path in enumerate(test_image_paths):
    img_rgb, img_float, input_tensor = load_for_cam(img_path)
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(i)])
    visualization = show_cam_on_image(img_float, grayscale_cam[0], use_rgb=True)

    axes[0][i].imshow(img_rgb); axes[0][i].set_title(CLASSES[i], fontsize=10); axes[0][i].axis('off')
    axes[1][i].imshow(visualization); axes[1][i].set_title('Grad-CAM', fontsize=10); axes[1][i].axis('off')

plt.suptitle('Grad-CAM Explainability — All Disease Classes', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/gradcam_multi.png', dpi=300)
plt.show()


## 9. Confidence Score Distribution

Separates confidence scores for correct vs. incorrect predictions — the basis for the clinical triage threshold analysis in the paper.

In [ ]:
confidences_correct, confidences_wrong = [], []
for i in range(0, len(all_image_paths), batch_size):
    batch_paths = all_image_paths[i:i + batch_size]
    batch_labels = all_true_labels[i:i + batch_size]
    batch_results = model.predict(source=batch_paths, save=False, verbose=False)
    for r, true_label in zip(batch_results, batch_labels):
        conf = r.probs.top1conf.item()
        if r.probs.top1 == true_label:
            confidences_correct.append(conf)
        else:
            confidences_wrong.append(conf)

plt.figure(figsize=(12, 5))
plt.hist(confidences_correct, bins=50, alpha=0.7, color='#2ecc71', label='Correct Predictions')
plt.hist(confidences_wrong, bins=50, alpha=0.7, color='#e74c3c', label='Wrong Predictions')
plt.title('Confidence Score Distribution', fontsize=14)
plt.xlabel('Confidence Score', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/confidence_distribution.png', dpi=300)
plt.show()


## 10. Comparison with Baseline Models

(Baseline figures from identical experimental conditions, reported in the paper.)

In [ ]:
table_data = [
    ['YOLO11n (Ours)', 'ISIC 2019', '8', f'{metrics.top1*100:.2f}%', f'{metrics.top5*100:.2f}%', '3.2 GFLOPs'],
    ['YOLOv8n (2023)', 'ISIC 2019', '8', '79.10%', '98.20%', '8.7 GFLOPs'],
    ['ResNet50 (2022)', 'ISIC 2019', '8', '78.50%', '97.80%', '25.6 GFLOPs'],
    ['GoogleNet (2021)', 'ISIC 2019', '8', '76.30%', '96.50%', '15.3 GFLOPs'],
]
columns = ['Model', 'Dataset', 'Classes', 'Top-1 Acc', 'Top-5 Acc', 'Complexity']

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')
table = ax.table(cellText=table_data, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2)
for j in range(len(columns)):
    table[1, j].set_facecolor('#d4edda')
    table[1, j].set_text_props(fontweight='bold')

plt.title('Comparison With Existing Methods', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


## 11. Save Final Results Summary

In [ ]:
per_class_lines = "\n".join(
    f"{cls:5s}: {acc:.1f}%" for cls, acc in zip(CLASSES, per_class_acc)
)

results_text = f"""YOLO11 SKIN DISEASE DETECTION - COMPLETE RESULTS
================================================================

DATASET: ISIC 2019
Total Images: {len(df)}
Classes: {', '.join(CLASSES)}
Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}

MODEL: YOLO11n-cls
Training Epochs: 50
Image Size: 224x224

MAIN RESULTS:
Top-1 Accuracy: {metrics.top1*100:.2f}%
Top-5 Accuracy: {metrics.top5*100:.2f}%

PER-CLASS ACCURACY:
{per_class_lines}

COMPARISON WITH OTHER MODELS:
YOLO11n (Ours): {metrics.top1*100:.2f}% Top-1, {metrics.top5*100:.2f}% Top-5, 3.2 GFLOPs
YOLOv8n (2023): 79.10% Top-1, 98.20% Top-5, 8.7 GFLOPs
ResNet50 (2022): 78.50% Top-1, 97.80% Top-5, 25.6 GFLOPs
GoogleNet (2021): 76.30% Top-1, 96.50% Top-5, 15.3 GFLOPs
"""

with open(f'{DRIVE_ROOT}/FINAL_RESULTS.txt', 'w') as f:
    f.write(results_text)

print(results_text)
print("✅ Results saved to Drive as FINAL_RESULTS.txt")
